# PEFT vs prompting (primary model)

Compares the fine-tuned (QLoRA) Qwen models with zero-shot and five-shot prompting of the same base model.
Everything is scored by the same code (`parse_prediction` + `compute_metrics`): prompting by
`run_prompting.py`, PEFT by `rescore_predictions.py`.

1. checks (all files present, same texts, same demonstration IDs)
2. main table
3. paired bootstrap: which differences are real?
4. per-emotion F1 for all conditions
5. label counts and label distribution
6. which texts are hard for every condition
7. Swedish texts without å/ä/ö

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from llm_project.prompting import LABELS, DEFAULT_VARIANT

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load((PROJECT_ROOT / "configs" / "prompting.yaml").read_text(encoding="utf-8"))

SPLIT = "dev"
MODEL = cfg.get("models", [{}])[0].get("name", "Qwen2.5-3B-Instruct").split("/")[-1]
PEFT_EXPERIMENT = "baseline_qlora_qv"
ID_KEY = cfg.get("prompting", {}).get("id_key", "alignment_id")

LABELS = LABELS
CONDITIONS = {
    "zero-shot": ("prompting", "zero_shot", "default"),
    "five-shot": ("prompting", "five_shot", "default"),
    "PEFT (EN-trained)": ("peft", "peft-english", PEFT_EXPERIMENT),
    "PEFT (SV-trained)": ("peft", "peft-target", PEFT_EXPERIMENT),
}

PRED = PROJECT_ROOT / "outputs" / "predictions"
METRICS = PROJECT_ROOT / "outputs" / "metrics"
pd.set_option("display.precision", 3)

LANGS = ["en", cfg.get("language", "sv")]

## 1. Checks

In [ ]:
from llm_project.notebook_utils import load_preds, load_metrics

def run_name(lang, cond):
    family, c, v = CONDITIONS[cond]
    return family, c, v

def try_load_preds(lang, cond):
    family, c, v = run_name(lang, cond)
    path_dir = PRED / family / SPLIT
    path = path_dir / f"{MODEL}__{lang}__{c}__{v}.jsonl"
    if not path.exists():
        return None
    return load_preds(MODEL, lang, c, path_dir, variant=v).set_index(ID_KEY)

def try_load_metrics(lang, cond):
    family, c, v = run_name(lang, cond)
    path_dir = METRICS / family / SPLIT
    path = path_dir / f"{MODEL}__{lang}__{c}__{v}.json"
    return load_metrics(MODEL, lang, c, path_dir, variant=v) if path.exists() else None

available = {(l, c): try_load_preds(l, c) for l in LANGS for c in CONDITIONS}
available = {k: v for k, v in available.items() if v is not None}
print("Runs found:")
for (l, c), df in available.items():
    print(f"  {l}  {c:18s} n={len(df)}")
print("Not found (expected: PEFT (SV-trained) on English):",
      [f"{l}/{c}" for l in LANGS for c in CONDITIONS if (l, c) not in available])

for l in LANGS:
    ids = [set(df.index) for (ll, _), df in available.items() if ll == l]
    assert all(s == ids[0] for s in ids), f"{l}: runs cover different texts"

fs = [try_load_metrics(l, "five-shot") for l in LANGS]
fs = [m for m in fs if m]
print("five-shot IDs:", {tuple(m["five_shot_ids"]) for m in fs})
print("fingerprints:", {m.get("fingerprint") for m in fs})
assert len({tuple(m["five_shot_ids"]) for m in fs}) == 1, "five-shot runs used different demonstration IDs"

## 2. Main table
Primary metric: macro-F1. `labels/text` should be compared with the gold average.

In [ ]:
rows = []
for (l, c) in available:
    m = load_metrics(l, c)
    rows.append({"lang": l, "condition": c, "macro_f1": m["macro_f1"], "micro_f1": m["micro_f1"],
                 "samples_f1": m["samples_f1"], "jaccard": m["jaccard"], "exact_match": m["exact_match"],
                 "malformed": m["malformed_rate"], "empty": m["empty_prediction_rate"],
                 "labels/text": m["pred_label_cardinality"], "gold labels/text": m["gold_label_cardinality"]})
main = pd.DataFrame(rows)
order = {c: i for i, c in enumerate(CONDITIONS)}
main = main.sort_values(["lang", "condition"], key=lambda s: s.map(order) if s.name == "condition" else s)
display(main.set_index(["lang", "condition"]))

ax = main.pivot(index="condition", columns="lang", values="macro_f1").reindex(list(CONDITIONS)).plot.bar(rot=0, figsize=(8, 4))
ax.set_ylabel("macro-F1"); ax.set_title(f"Qwen2.5-3B-Instruct, {SPLIT}"); plt.tight_layout(); plt.show()

## 3. Paired bootstrap

1,000 resamples of the texts; both conditions are scored on the *same* resampled texts (matched by
alignment ID). If the 95% interval excludes 0, the difference is unlikely to be noise.
The cross-lingual rows compare English and Swedish versions of the same aligned texts.

In [ ]:
from llm_project.notebook_utils import binarise, f1s, paired_bootstrap

comparisons = []
for l in LANGS:
    for peft in ["PEFT (EN-trained)", "PEFT (SV-trained)"]:
        for base in ["zero-shot", "five-shot"]:
            comparisons.append((f"{peft} vs {base}", (l, base), (l, peft)))
comparisons.append(("SV-trained vs EN-trained (on SV)", ("sv", "PEFT (EN-trained)"), ("sv", "PEFT (SV-trained)")))
for c in CONDITIONS:
    comparisons.append((f"SV vs EN | {c}", ("en", c), ("sv", c)))

rows = []
for name, a, b in comparisons:
    if a not in available or b not in available:
        continue
    d, lo, hi, n = paired_bootstrap(available[a], available[b], labels=LABELS)
    rows.append({"comparison": name, "lang (A -> B)": f"{a[0]} -> {b[0]}", "macro_f1 diff (B-A)": d,
                 "ci_low": lo, "ci_high": hi, "significant": not (lo <= 0 <= hi), "n": n})
boot = pd.DataFrame(rows)
boot

## 4. Per-emotion F1
Which emotions does fine-tuning improve, and which transfer from English to Swedish?

In [ ]:
per = {}
for (l, c) in available:
    m = load_metrics(l, c)
    per[(l, c)] = {e: m["per_label"][e]["f1"] for e in LABELS}
per = pd.DataFrame(per)[[k for l in LANGS for k in [(l, c) for c in CONDITIONS] if k in available]]
first = next(iter(available))
per["gold_support_sv"] = [load_metrics("sv", "zero-shot")["per_label"][e]["support"] for e in LABELS] \
    if ("sv", "zero-shot") in available else np.nan
display(per)

f1_only = per.drop(columns="gold_support_sv")
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(f1_only.values, cmap="viridis", vmin=0, vmax=max(0.6, np.nanmax(f1_only.values)))
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xticks(range(f1_only.shape[1]), [f"{l}\n{c}" for l, c in f1_only.columns], fontsize=8)
for i in range(f1_only.shape[0]):
    for j in range(f1_only.shape[1]):
        ax.text(j, i, f"{f1_only.iat[i, j]:.2f}", ha="center", va="center", fontsize=7, color="w")
fig.colorbar(im, label="F1"); ax.set_title("Per-emotion F1"); plt.tight_layout(); plt.show()

## 5. Label counts and label distribution
How often each emotion is predicted, against how often it is gold.

In [ ]:
for l in LANGS:
    runs = {c: available[(l, c)] for c in CONDITIONS if (l, c) in available}
    if not runs:
        continue
    gold = next(iter(runs.values()))["gold"]
    table = {"GOLD": pd.Series([e for g in gold for e in g]).value_counts().reindex(LABELS, fill_value=0)}
    for c, df in runs.items():
        table[c] = pd.Series([e for p in df["pred"] for e in p]).value_counts().reindex(LABELS, fill_value=0)
    print(f"--- {l}: predicted counts per emotion ---")
    display(pd.DataFrame(table))

## 6. Hard texts: wrong for every condition
Texts that no condition gets exactly right are often texts without an emotional cue (context lost from the film).

In [ ]:
for l in LANGS:
    runs = {c: available[(l, c)] for c in CONDITIONS if (l, c) in available}
    if len(runs) < 2:
        continue
    ids = next(iter(runs.values())).index
    exact = pd.DataFrame({c: [set(df.at[i, "gold"]) == set(df.at[i, "pred"]) for i in ids] for c, df in runs.items()}, index=ids)
    print(f"--- {l}: texts exactly right per number of conditions ---")
    print(exact.sum(axis=1).value_counts().sort_index().to_string())
    hard = exact.index[~exact.any(axis=1)]
    print(f"{len(hard)} of {len(ids)} texts are wrong in every condition. Sample:")
    first_df = next(iter(runs.values()))
    with pd.option_context("display.max_colwidth", 90):
        display(first_df.loc[hard[:10], ["text", "gold"]].assign(**{c: runs[c].loc[hard[:10], "pred"] for c in runs}))

## 7. Swedish texts without å/ä/ö
Some subtitles lost their diacritics (e.g. *grata* for *gråta*). Does that hurt accuracy?

In [ ]:
if ("sv", "zero-shot") in available:
    sv_text = available[("sv", "zero-shot")]["text"]
    no_diacritics = ~sv_text.str.lower().str.contains("[åäö]")
    print(f"{no_diacritics.sum()} of {len(sv_text)} Swedish texts contain no å/ä/ö "
          "(includes texts that simply don't need them)")
    rows = []
    for c in CONDITIONS:
        if ("sv", c) not in available:
            continue
        df = available[("sv", c)]
        for flag, label in [(no_diacritics, "no å/ä/ö"), (~no_diacritics, "has å/ä/ö")]:
            sub = df[flag.reindex(df.index).values]
            rows.append({"condition": c, "group": label, "n": len(sub),
                         "macro_f1": f1s(binarise(sub["gold"]), binarise(sub["pred"]))[0]})
    display(pd.DataFrame(rows).pivot(index="condition", columns="group", values="macro_f1"))
    with pd.option_context("display.max_colwidth", 90):
        display(sv_text[no_diacritics].head(20).to_frame())